In [ ]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{\r\n  "username": "YOUR_KAGGLE_USERNAME",\r\n  "key": "KGAT_2957138dac87ff071a98db5d2aad8c79"\r\n}'}

In [ ]:
!pip install -q kaggle

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json


In [ ]:
!kaggle datasets download -d bjoernjostein/physionet-challenge-2016 -p /content
!unzip -q /content/physionet-challenge-2016.zip -d /content/physionet2016


Dataset URL: https://www.kaggle.com/datasets/bjoernjostein/physionet-challenge-2016
License(s): ODC Attribution License (ODC-By)
 87% 180M/207M [00:00<00:00, 496MB/s]
100% 207M/207M [00:00<00:00, 516MB/s]


In [ ]:
!pip install -q tensorflow librosa scipy scikit-learn pandas


In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import librosa
import tensorflow as tf
from scipy.signal import find_peaks
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.utils import class_weight


In [ ]:
SR = 4000
DURATION = 5
N_MFCC = 40
MAX_LEN = 216


In [ ]:
def extract_mfcc(file):
    audio, _ = librosa.load(file, sr=SR, duration=DURATION)

    mfcc = librosa.feature.mfcc(y=audio, sr=SR, n_mfcc=N_MFCC)
    mfcc = mfcc.T

    if mfcc.shape[0] < MAX_LEN:
        mfcc = np.pad(mfcc, ((0, MAX_LEN - mfcc.shape[0]), (0, 0)))
    else:
        mfcc = mfcc[:MAX_LEN]

    return mfcc


In [ ]:
X = []
y = []

base_path = "/content/physionet2016"

reference_files = glob.glob(base_path + "/**/REFERENCE.csv", recursive=True)
print("Found reference files:", reference_files)

for ref_file in reference_files:
    df = pd.read_csv(ref_file, header=None)
    folder = os.path.dirname(ref_file)

    for _, row in df.iterrows():
        file_id = row[0]
        label = row[1]
        wav_path = os.path.join(folder, file_id + ".wav")

        if not os.path.exists(wav_path):
            continue

        X.append(extract_mfcc(wav_path))

        if label == -1:
            y.append(0)  # Normal
        else:
            y.append(1)  # Abnormal

X = np.array(X)[..., np.newaxis]
y = np.array(y)

print("Total samples:", X.shape[0])
print("Input shape:", X.shape)


Found reference files: ['/content/physionet2016/validation/REFERENCE.csv', '/content/physionet2016/training-e/REFERENCE.csv', '/content/physionet2016/training-f/REFERENCE.csv', '/content/physionet2016/training-d/REFERENCE.csv', '/content/physionet2016/training-a/REFERENCE.csv', '/content/physionet2016/training-c/REFERENCE.csv', '/content/physionet2016/training-b/REFERENCE.csv']
Total samples: 3541
Input shape: (3541, 216, 40, 1)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [ ]:
class_weights = {
    0: 1.0,
    1: 2.0   # increase abnormal importance
}
print("Class weights:", class_weights)


Class weights: {0: 1.0, 1: 2.0}


In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, (3,3), activation='relu', input_shape=X.shape[1:]),
    tf.keras.layers.MaxPooling2D((2,2)),

    tf.keras.layers.Conv2D(64, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2,2)),

    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.3),

    tf.keras.layers.Dense(2, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_2 (Conv2D)               │ (None, 214, 38, 32)    │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 107, 19, 32)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 105, 17, 64)    │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 52, 8, 64)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 26624)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │     3,408,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 2)              │           258 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,427,074 (13.07 MB)

 Trainable params: 3,427,074 (13.07 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=30,
    batch_size=32,
    class_weight=class_weights
)


Epoch 1/30
89/89 ━━━━━━━━━━━━━━━━━━━━ 7s 43ms/step - accuracy: 0.6582 - loss: 5.5021 - val_accuracy: 0.8420 - val_loss: 0.3535
Epoch 2/30
89/89 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.8444 - loss: 0.4617 - val_accuracy: 0.8350 - val_loss: 0.3266
Epoch 3/30
89/89 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.8617 - loss: 0.4000 - val_accuracy: 0.8646 - val_loss: 0.3019
Epoch 4/30
89/89 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.8557 - loss: 0.3989 - val_accuracy: 0.8477 - val_loss: 0.3387
Epoch 5/30
89/89 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.8781 - loss: 0.3632 - val_accuracy: 0.8759 - val_loss: 0.2829
Epoch 6/30
89/89 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.8856 - loss: 0.3172 - val_accuracy: 0.8364 - val_loss: 0.3381
Epoch 7/30
89/89 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9013 - loss: 0.3122 - val_accuracy: 0.8491 - val_loss: 0.3128
Epoch 8/30
89/89 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9049 - loss: 0.2869 - val_accuracy: 0.8759 - v

In [ ]:
y_pred = np.argmax(model.predict(X_test), axis=1)

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["Normal", "Abnormal"]))


23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step
Confusion Matrix:
[[479  67]
 [ 24 139]]

Classification Report:
              precision    recall  f1-score   support

      Normal       0.95      0.88      0.91       546
    Abnormal       0.67      0.85      0.75       163

    accuracy                           0.87       709
   macro avg       0.81      0.87      0.83       709
weighted avg       0.89      0.87      0.88       709



In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]
tflite_model = converter.convert()

with open("heart_model.tflite", "wb") as f:
    f.write(tflite_model)

print("TFLite model saved")


Saved artifact at '/tmp/tmpsqsaq5vs'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 216, 40, 1), dtype=tf.float32, name='keras_tensor_9')
Output Type:
  TensorSpec(shape=(None, 2), dtype=tf.float32, name=None)
Captures:
  134901896619088: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134901896625232: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134901896623696: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134901896621200: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134901896621392: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134901896614864: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134901896617744: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134901896621776: TensorSpec(shape=(), dtype=tf.resource, name=None)
TFLite model saved


In [ ]:
from google.colab import files
files.download("heart_model.tflite")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>